In [1]:
!pip install openai requests pillow


In [2]:
import openai, requests, base64, os, json, time
print(f"openai: {openai.__version__}")
print("Imports OK")


openai: 2.32.0
Imports OK


# NeurIPS Computational Resources & Reproducibility Metadata

Required by **NeurIPS 2025 Paper Checklist §8**. Run to print a live hardware/version summary. Fill `[TODO]` fields before submission.

In [3]:
"""
NeurIPS 2025 Checklist S8 - Computational Resources
====================================================
Hardware
  GPU      : NVIDIA RTX A6000 (48 GB VRAM)
  CPU      : [TODO: e.g. AMD EPYC 7542 32-core]
  RAM      : [TODO: e.g. 256 GB DDR4]
  OS       : Windows 11 / Ubuntu 22.04
  Provider : Local on-premise workstation

Model & Inference
  Model      : gpt-5.5
  max_completion_tokens : 4096 per call
  Temp       : 0.0 (Zero-Shot / Sequential / LtM / ReAct / CoT)
               0.1 (Iterative)
               0.1-0.5 (Self-Consistency runs)
               0.1/0.7 (Meta-Prompting: analysis/generation)

API Calls per Video  (C = ceil(frames / 10))
  Zero-Shot        : C
  Sequential       : 5*C + 1
  Least-To-Most    : 8*C + 1
  ReAct            : C + 1
  True Iterative   : up to 8*C
  Self-Consistency : 5*C + 1
  Meta-Prompting   : 2*C + 2
  Chain-of-Thought : C + 1
  Total/video (50 frames, C=5) ~ 162 calls ~ 243 000 tokens

Total Compute (fill before submission)
  Dataset  : [TODO] videos x [TODO] avg frames
  Calls    : [TODO] x 162 = [TODO]
  Tokens   : [TODO] x 243 000 = [TODO]
  Time     : ~[TODO] hours

Reproducibility
  - Checkpoint files save progress after every video (atomic write)
  - Re-running any cell after failure resumes with zero extra API cost
  - All raw API responses saved as JSON before post-processing
  - Chunk-level saves after every 10-frame batch
"""
import subprocess, platform, datetime
print("=" * 64)
print(f"  NeurIPS Compute Summary  - {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}")
print("=" * 64)
print(f"  Python  : {platform.python_version()}")
print(f"  OS      : {platform.platform()}")
try:
    smi = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        text=True).strip()
    for line in smi.split("\n"):
        print(f"  GPU     : {line.strip()}")
except Exception:
    print("  GPU     : nvidia-smi not available")
try:
    import openai
    print(f"  openai  : {openai.__version__}")
except Exception:
    pass
print(f"  Model   : gpt-5.5")
print(f"  Chunks  : 10 frames/chunk  |  max_completion_tokens : 4096")
print(f"  Retry   : 7 attempts, exponential back-off, cap 5 min")
print("=" * 64)


  NeurIPS Compute Summary  - 2026-04-28 22:49
  Python  : 3.10.11
  OS      : Windows-10-10.0.26100-SP0
  GPU     : NVIDIA H100 NVL, 95830 MiB
  GPU     : NVIDIA H100 NVL, 95830 MiB
  GPU     : NVIDIA H100 NVL, 95830 MiB
  GPU     : NVIDIA H100 NVL, 95830 MiB
  openai  : 2.32.0
  Model   : gpt-5.5
  Chunks  : 10 frames/chunk  |  max_completion_tokens : 4096
  Retry   : 7 attempts, exponential back-off, cap 5 min


#Iterative prompting
Iterative Prompting Approach
The Iterative Prompting technique follows a structured refinement process:

- Initial Analysis: The model provides a first-pass analysis of what appears to be happening in the frames
- Guided Iterations: Through a series of targeted follow-up prompts, the model refines specific aspects of its analysis
- Progressive Improvement: Each round builds on previous insights while addressing potential weaknesses or gaps
- Final Synthesis: After multiple refinement rounds, the model creates a final, comprehensive assessment

Implementation Highlights

Multi-Round Refinement:

Starts with an initial general analysis prompt
Follows with 4 specialized refinement rounds:

- People and relationships focus
- Actions and intent focus
- Criminal elements and evidence focus
- Critical examination (missing elements, alternative interpretations)


Concludes with a final synthesis prompt for each chunk


Complete Frame Processing:

- Processes all frames in chunks of 10 frames each
- Each chunk undergoes the full iterative process independently


Conversation Continuity:

- Maintains the complete conversation history throughout all rounds
- Each refinement builds on the accumulated context from previous rounds
- Creates a progressive improvement cycle where later responses incorporate earlier insights


Holistic Synthesis:

- After all chunks are iteratively analyzed, performs a final cross-chunk synthesis
- Creates a coherent narrative of the entire incident
- Addresses any discrepancies between chunk analyses

In [4]:
import os
import json
import base64
import requests
import time
from datetime import datetime
from collections import defaultdict


import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
# ================================================================
#  CONFIGURATION  -  edit these paths to match your machine
# ================================================================
FRAMES_DIR     = r"C:\Opeyemi\PROMPTS\FRAMES"   # pre-extracted frames
RESULTS_BASE   = r"C:\Opeyemi\PROMPTS\RESULTS"  # all JSON outputs
FRAME_EXT      = ".jpg"
FRAME_INTERVAL = 1   # 1=every frame; 2=every other; etc.
BATCH_SIZE     = 20   # max frames per API call
MAX_WORKERS    = 4    # parallel videos processed at once (OpenAI rate limits depend on tier; raise carefully)

# ================================================================
#  FRAME HELPERS  -  self-contained in every cell
# ================================================================

def extract_frame_number(filename):
    """Return integer index from frame_00042.jpg style names."""
    import re as _re
    name = os.path.splitext(filename)[0]
    m = _re.search(r"frame[_\-]?(\d+)", name, _re.IGNORECASE)
    if m:
        return int(m.group(1))
    nums = _re.findall(r"\d+", name)
    return int(nums[-1]) if nums else 0


def discover_all_videos_and_frames(frames_dir=None):
    """
    Walk FRAMES_DIR and return a manifest of all extracted videos.

    Expected layout (created by the Video-to-Frames extractor):
        FRAMES_DIR/
            Abuse/
                Abuse001_x264/
                    frame_00001.jpg
                    frame_00002.jpg ...
                Abuse002_x264/ ...
            Arrest/ ...   (13 UCF-Crime categories)

    Returns dict "<CrimeType>_<VideoStem>" -> {
        "crime_type": "Abuse",
        "video_id":   "Abuse001_x264",
        "frames_dir": r"C:\...\FRAMES\Abuse\Abuse001_x264",
        "frames":     ["frame_00001.jpg", ...]   # sorted by number
    }
    """
    if frames_dir is None:
        frames_dir = FRAMES_DIR
    print(f"\n=== DISCOVERING FRAMES ===")
    print(f"    Root : {frames_dir}")
    all_videos = {}
    if not os.path.isdir(frames_dir):
        print(f"  ERROR: FRAMES_DIR not found: {frames_dir}")
        print("  Run the Video-to-Frames extractor first, or check the path.")
        return all_videos
    crime_types = sorted([
        d for d in os.listdir(frames_dir)
        if os.path.isdir(os.path.join(frames_dir, d)) and not d.startswith("_")
    ])
    print(f"  Categories : {crime_types}")
    for crime_type in crime_types:
        cat_dir = os.path.join(frames_dir, crime_type)
        video_stems = sorted([
            d for d in os.listdir(cat_dir)
            if os.path.isdir(os.path.join(cat_dir, d))
        ])
        print(f"    {crime_type:20s}: {len(video_stems)} videos")
        for video_stem in video_stems:
            vdir = os.path.join(cat_dir, video_stem)
            frame_files = sorted(
                [ff for ff in os.listdir(vdir) if ff.lower().endswith(FRAME_EXT)],
                key=extract_frame_number
            )
            if not frame_files:
                print(f"      WARNING: no {FRAME_EXT} frames in {vdir} - skipping")
                continue
            key = f"{crime_type}_{video_stem}"
            all_videos[key] = {
                "crime_type" : crime_type,
                "video_id"   : video_stem,
                "frames_dir" : vdir,
                "frames"     : frame_files,
            }
    print(f"  Total videos ready: {len(all_videos)}")
    return all_videos


def load_frames_for_video(video_info, frame_interval=1):
    """Read every frame_interval-th .jpg, base64-encode, return dict."""
    frames_data = {}
    vdir        = video_info["frames_dir"]
    frame_files = video_info["frames"]
    video_id    = video_info["video_id"]
    selected    = frame_files[::frame_interval]
    label = "ALL" if frame_interval == 1 else f"every {frame_interval}th"
    print(f"  Loading {len(selected)} frames ({label}) for {video_id} ...")
    for ff in selected:
        fp = os.path.join(vdir, ff)
        try:
            with open(fp, "rb") as fh:
                frames_data[ff] = base64.b64encode(fh.read()).decode("utf-8")
        except Exception as e:
            print(f"    ERROR loading {ff}: {e}")
    print(f"  Loaded {len(frames_data)}/{len(selected)} frames OK")
    return frames_data

SAVE_DIR = r"C:\Opeyemi\PROMPTS\RESULTS\GPT\TRUE-ITERATIVE"
os.makedirs(SAVE_DIR, exist_ok=True)


CHECKPOINT_FILE = os.path.join(SAVE_DIR, "true_iterative_checkpoint.json")


def load_checkpoint():
    """Resume from last completed video after any crash or network failure."""
    if os.path.exists(CHECKPOINT_FILE):
        try:
            with open(CHECKPOINT_FILE, "r") as f:
                data = json.load(f)
            n = len(data.get("completed_videos", []))
            print(f"  Checkpoint: {n} videos already done - skipping them.")
            return data
        except Exception as e:
            print(f"  Could not read checkpoint ({e}) - starting fresh.")
    return {"completed_videos": [], "results": {}}


def save_checkpoint(data):
    """Atomic write so the file is never corrupted on a crash."""
    os.makedirs(SAVE_DIR, exist_ok=True)
    tmp = CHECKPOINT_FILE + ".tmp"
    with open(tmp, "w") as f:
        json.dump(data, f, indent=2)
    os.replace(tmp, CHECKPOINT_FILE)








def _build_image_content(frames_data):
    """Convert frames_data dict into GPT vision content blocks, BATCH_SIZE at a time."""
    frame_names = sorted(frames_data.keys(), key=lambda x: extract_frame_number(x))
    batches = [frame_names[i:i+BATCH_SIZE] for i in range(0, len(frame_names), BATCH_SIZE)]
    return batches, frame_names


def _make_payload(api_key, messages):
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }
    payload = {
        "model": "gpt-5.5",
        "messages": messages,
        "max_completion_tokens": 4096
    }
    return headers, payload


def _frames_to_content(frames_data, frame_names):
    """Build a GPT vision content list from a batch of frame names."""
    content = []
    for fname in frame_names:
        b64 = frames_data[fname]
        content.append({
            "type": "image_url",
            "image_url": {
                "url": f"data:image/jpeg;base64,{b64}",
                "detail": "low"
            }
        })
        content.append({"type": "text", "text": f"[Frame: {fname}]"})
    return content


MAX_ITERATIONS = 4

class TrueIterativeGPTAnalyzer:
    """
    True Iterative crime analysis using GPT-5.5 with BATCH_SIZE frame batching.
    Iteration 1 processes frames in batches (images); subsequent iterations refine text-only.
    """
    def __init__(self, api_key):
        self.api_key = api_key

    def _text_call(self, messages):
        headers, payload = _make_payload(self.api_key, messages)
        result = make_gpt_request_robust(headers, payload)
        return result.get("choices",[{}])[0].get("message",{}).get("content", str(result))

    @staticmethod
    def _extract_classification(text):
        import re
        for line in text.splitlines():
            if line.strip().upper().startswith("CLASSIFICATION"):
                return line.strip().upper()
        return ""

    def analyze_frames(self, frames_data, video_id, crime_type):
        print(f"\n  [True-Iterative] {video_id} | {len(frames_data)} frames | max {MAX_ITERATIONS} iterations ...")
        frame_names = sorted(frames_data.keys(), key=lambda x: extract_frame_number(x))
        batches = [frame_names[i:i+BATCH_SIZE] for i in range(0, len(frame_names), BATCH_SIZE)]
        iteration_log = {}

        # Iteration 1: Batched image analysis
        print(f"    Iteration 1 (images): {len(batches)} batches ...")
        batch_summaries = []
        for idx, batch in enumerate(batches, 1):
            content = _frames_to_content(frames_data, batch)
            content.append({"type": "text", "text": (
                f"Iteration 1 analysis - Batch {idx}/{len(batches)}: "
                "CLASSIFICATION: [crime type]\nCONFIDENCE: [0-100%]\n"
                "KEY EVIDENCE: [main evidence]\nUNCERTAINTIES: [what is unclear]"
            )})
            messages = [{"role": "user", "content": content}]
            headers, payload = _make_payload(self.api_key, messages)
            result = make_gpt_request_robust(headers, payload)
            s = result.get("choices",[{}])[0].get("message",{}).get("content", str(result))
            batch_summaries.append(s)

        formatted = "\n\n".join(f"--- Batch {i+1} ---\n{s}" for i,s in enumerate(batch_summaries))
        synth = self._text_call([{"role": "user", "content": (
            f"Synthesize {len(batches)} batch analyses of {len(frames_data)} frames:\n{formatted}\n\n"
            "CLASSIFICATION: [crime type]\nCONFIDENCE: [0-100%]\nKEY EVIDENCE: [summary]\nUNCERTAINTIES: [what is unclear]"
        )}])
        iteration_log["iteration_1"] = synth
        iteration_log["iteration_1_batch_summaries"] = batch_summaries
        print(f"      First answer: {len(synth)} chars")

        prev = synth
        converged = False
        iterations_run = 1

        for iteration in range(2, MAX_ITERATIONS + 1):
            print(f"    Iteration {iteration} (text refinement) ...")
            refined = self._text_call([{"role": "user", "content": (
                f"ITERATION {iteration} - Refine your analysis.\n\nPrevious answer (Iteration {iteration-1}):\n{prev}\n\n"
                "Critically review: Is classification still best? Any under/over-weighted evidence? "
                "Can you resolve uncertainties?\n\n"
                "CLASSIFICATION: [crime type]\nCONFIDENCE: [0-100%]\nKEY EVIDENCE: [updated]\n"
                "UNCERTAINTIES: [remaining]\nCHANGES FROM PREVIOUS: [what changed and why]"
            )}])
            iteration_log[f"iteration_{iteration}"] = refined
            print(f"      Response: {len(refined)} chars")

            if (self._extract_classification(prev) == self._extract_classification(refined)
                    and self._extract_classification(refined)):
                print(f"      Converged at iteration {iteration}!")
                converged = True
                iterations_run = iteration
                prev = refined
                break
            prev = refined
            iterations_run = iteration

        final = self._text_call([{"role": "user", "content": (
            f"FINAL REPORT after {iterations_run} iteration(s).\n\nFinal answer:\n{prev}\n\n"
            "PRIMARY CLASSIFICATION: [crime type]\nCONFIDENCE LEVEL: [0-100%]\nSEVERITY: [Low/Medium/High/Critical]\n"
            "KEY EVIDENCE:\n- [point 1]\n- [point 2]\nCONVERGENCE SUMMARY: [iterations and changes]\n"
            "ALTERNATIVE INTERPRETATIONS: [other explanations]\nRECOMMENDED LAW ENFORCEMENT RESPONSE: [actions]"
        )}])
        iteration_log["final_report"] = final

        return {
            "video_id": video_id, "crime_type": crime_type,
            "frames_analyzed": len(frames_data), "total_batches": len(batches),
            "batch_size": BATCH_SIZE, "prompting_technique": "TRUE-ITERATIVE",
            "model": "gpt-5.5", "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
            "convergence_summary": {"converged": converged, "total_iterations_run": iterations_run, "max_iterations": MAX_ITERATIONS},
            "iteration_log": iteration_log
        }



def make_gpt_request_robust(headers, payload, max_retries=7, base_wait=5):
    """
    Fault-tolerant OpenAI API call using requests.
    Retries on: rate limits (429), network errors, server errors (5xx).
    Exponential back-off with jitter, capped at 5 minutes.
    Returns parsed JSON on success, or error-string dict on permanent failure.
    """
    import random
    url = "https://api.openai.com/v1/chat/completions"

    for attempt in range(1, max_retries + 1):
        try:
            response = requests.post(url, headers=headers, json=payload, timeout=120)

            if response.status_code == 200:
                return response.json()

            # Rate limit
            if response.status_code == 429:
                wait = base_wait * (2 ** (attempt - 1)) + random.uniform(0, 2)
                wait = min(wait, 300)
                print(f"[Retry {attempt}/{max_retries}] 429 Rate limit. Waiting {wait:.1f}s ...")
                time.sleep(wait)
                continue

            # Server errors – retry
            if response.status_code >= 500:
                wait = base_wait * (2 ** (attempt - 1)) + random.uniform(0, 2)
                wait = min(wait, 300)
                print(f"[Retry {attempt}/{max_retries}] HTTP {response.status_code}. Waiting {wait:.1f}s ...")
                time.sleep(wait)
                continue

            # Permanent client errors (4xx except 429) – do not retry
            try:
                err = response.json()
            except Exception:
                err = response.text
            print(f"[FATAL] HTTP {response.status_code}: {err}")
            return {"error": f"HTTP_{response.status_code}", "detail": str(err)}

        except (requests.exceptions.ConnectionError,
                requests.exceptions.Timeout,
                requests.exceptions.ChunkedEncodingError) as e:
            wait = base_wait * (2 ** (attempt - 1)) + random.uniform(0, 2)
            wait = min(wait, 300)
            print(f"[Retry {attempt}/{max_retries}] Network error: {type(e).__name__}: {e}")
            print(f"  Waiting {wait:.1f}s ...")
            time.sleep(wait)

        except Exception as e:
            if attempt < max_retries:
                print(f"[Retry {attempt}/{max_retries}] Unexpected: {type(e).__name__}: {e}")
                time.sleep(base_wait * attempt)
            else:
                return {"error": str(e)}

    return {"error": f"All {max_retries} attempts exhausted"}





def process_all_crime_folders(api_key):
    """
    Analyse every video in FRAMES_DIR.
    Already-completed videos are skipped (checkpoint/resume) so re-running
    after any failure picks up exactly where it stopped.
    """
    analyzer   = TrueIterativeGPTAnalyzer(api_key)
    all_videos = discover_all_videos_and_frames()
    if not all_videos:
        print("No videos found! Verify FRAMES_DIR path."); return {}
    cp          = load_checkpoint()
    all_results = cp.get("results", {})
    done_set    = set(cp.get("completed_videos", []))
    skipped     = []
    total       = len(all_videos)
    remaining   = {k: v for k, v in all_videos.items() if k not in done_set}
    print(f"\nVideos: total={total} | done={len(done_set)} | remaining={len(remaining)}")
    # ================================================================
    #  PARALLEL VIDEO PROCESSING
    # ================================================================
    checkpoint_lock = threading.Lock()
    print_lock      = threading.Lock()

    def _process_one_video(vkey, vinfo):
        """Process a single video in its own thread. Checkpoint writes are locked."""
        with print_lock:
            print(f"\n  [START] {vkey}")
        try:
            frames = load_frames_for_video(vinfo, FRAME_INTERVAL)
            if not frames:
                return vkey, None, "no frames"
            res = analyzer.analyze_frames(frames, vinfo["video_id"], vinfo["crime_type"])
            with checkpoint_lock:
                all_results[vkey] = res
                done_set.add(vkey)
                save_checkpoint({"completed_videos": list(done_set), "results": all_results})
            with print_lock:
                print(f"  [DONE]  {vkey}  ({len(done_set)}/{total})")
            return vkey, res, None
        except Exception as e:
            with print_lock:
                print(f"  [ERROR] {vkey}: {e}")
            with checkpoint_lock:
                save_checkpoint({"completed_videos": list(done_set), "results": all_results})
            return vkey, None, f"error: {e}"

    print(f"\n  Launching ThreadPoolExecutor with {MAX_WORKERS} parallel workers ...")
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [executor.submit(_process_one_video, k, v) for k, v in remaining.items()]
        for fut in as_completed(futures):
            vkey, _, err = fut.result()
            if err:
                skipped.append(f"{vkey} ({err})")

    ts = time.strftime("%Y%m%d_%H%M%S")
    summary = os.path.join(SAVE_DIR, f"true_iterative_summary_{ts}.json")
    with open(summary, "w") as f:
        json.dump(all_results, f, indent=2)
    if skipped:
        with open(os.path.join(SAVE_DIR, f"skipped_{ts}.txt"), "w") as f:
            f.write("\n".join(skipped))
    print(f"\nDone. Summary -> {summary}")
    print(f"  Processed: {len(all_results)} | Skipped: {len(skipped)}")
    return all_results


def run():
    # Load API key from file
    with open(r"C:\Opeyemi\PROMPTS\API-KEYS\chatgpt.txt", "r") as _f:
        api_key = _f.read().strip()

    """Main execution function"""
    print("TRUE Iterative Prompting Crime Video Analysis with GPT-5.5 - ALL Frames")
    print("="*70)
    print("TRUE ITERATIVE = Same question refined repeatedly until convergence")
    print("="*70)

    # Test directory access first
    print("Testing directory access...")
    for path in [FRAMES_DIR, SAVE_DIR]:
        print(f"Path: {path}")
        print(f"  Exists: {os.path.exists(path)}")
        if os.path.exists(path):
            try:
                contents = os.listdir(path)
                print(f"  Contains {len(contents)} items")
                if contents:
                    print(f"  First few items: {contents[:3]}")
            except Exception as e:
                print(f"  Error accessing contents: {str(e)}")

    # Get API key
    try:
        api_key_path = "C:\Opeyemi\PROMPTS\API-KEYS\chatgpt.txt"
        print(f"Trying to load API key from: {api_key_path}")
        print(f"File exists: {os.path.exists(api_key_path)}")

        f = open(api_key_path, "r")
        api_key = f.read().strip()
        f.close()

        if not api_key:
            print("✗ Failed to load GPT API key: File is empty")
            return

        print("✓ Successfully loaded GPT API key")
        print(f"API key starts with: {api_key[:5]}...")

    except Exception as e:
        print(f"✗ Failed to load GPT API key: {str(e)}")
        return

    # Verify directories exist
    print("\nVerifying directories:")
    print(f"Frames directory exists: {os.path.exists(FRAMES_DIR)}")
    print(f"Save directory exists: {os.path.exists(SAVE_DIR)}")

    if not os.path.exists(FRAMES_DIR):
        print(f"✗ Frames directory not found: {FRAMES_DIR}")
        return

    # Create save directory if it doesn't exist
    os.makedirs(SAVE_DIR, exist_ok=True)

    # Process all crime folders
    results = process_all_crime_folders(api_key)

    # Print summary
    total_frames_processed = 0
    total_videos_processed = len(results)
    total_iterations_run = 0
    convergence_achieved = 0

    for video_id, video_results in results.items():
        if video_results and 'convergence_summary' in video_results:
            summary = video_results['convergence_summary']
            total_frames_processed += video_results.get('frames_used', 0)
            total_iterations_run += summary.get('total_iterations_run', 0)
            if summary.get('converged', False):
                convergence_achieved += 1

    print("\n" + "="*70)
    print(f"TRUE ITERATIVE PROMPTING COMPLETE!")
    print(f"Videos processed: {total_videos_processed}")
    print(f"Total frames analyzed: {total_frames_processed}")
    print(f"Total iterations run: {total_iterations_run}")
    print(f"Convergence achieved: {convergence_achieved}/{total_videos_processed} videos")
    print(f"Model used: GPT-5.5")
    print(f"Method: Same core question refined repeatedly")
    print("="*70)

if __name__ == "__main__":
    run()

TRUE Iterative Prompting Crime Video Analysis with GPT-5.5 - ALL Frames
TRUE ITERATIVE = Same question refined repeatedly until convergence
Testing directory access...
Path: C:\Opeyemi\PROMPTS\FRAMES
  Exists: True
  Contains 13 items
  First few items: ['Abuse', 'Assault', 'Burglary']
Path: C:\Opeyemi\PROMPTS\RESULTS\GPT\TRUE-ITERATIVE
  Exists: True
  Contains 1 items
  First few items: ['true_iterative_checkpoint.json']
Trying to load API key from: C:\Opeyemi\PROMPTS\API-KEYS\chatgpt.txt
File exists: True
✓ Successfully loaded GPT API key
API key starts with: sk-pr...

Verifying directories:
Frames directory exists: True
Save directory exists: True

=== DISCOVERING FRAMES ===
    Root : C:\Opeyemi\PROMPTS\FRAMES
  Categories : ['Abuse', 'Assault', 'Burglary', 'Explosion', 'Fighting', 'RoadAccidents', 'Robbery', 'Shooting', 'Shoplifting', 'Stealing', 'Vandalism']
    Abuse               : 50 videos
    Assault             : 12 videos
    Burglary            : 100 videos
    Explosion

#Self-Consistency
This approach generates multiple independent analyses and determines the most reliable interpretation through consensus.
Self-Consistency Prompting Approach
The Self-Consistency technique follows a unique multi-analysis process:

Multiple Independent Analyses: The system generates several different analyses of the same frames
Diverse Perspectives: Each analysis uses a different prompt template to encourage varied viewpoints
Consensus Determination: The system identifies areas of agreement and disagreement across analyses
Confidence Assessment: For each key element, the level of consensus is explicitly evaluated

Implementation Highlights

Multi-Perspective Analysis:

Generates 5 independent analyses for each chunk of frames
Uses 5 distinct prompt templates to encourage diversity:

Standard analytical perspective
Forensic analyst perspective
Detective/law enforcement perspective
Security expert perspective
Witness testimony perspective


Higher temperature settings (0.5) for greater response diversity


Complete Frame Processing:

Processes all frames in chunks of 10 frames each
Each chunk undergoes the full multi-analysis process


Two-Level Consensus Building:

Chunk-Level Consensus: After generating multiple analyses for each chunk, determines consensus on:

Crime type
Perpetrator description
Victim description
Key actions
Evidence
Timeline


Cross-Chunk Consensus: After processing all chunks, synthesizes a final consensus across the entire video

In [5]:
import os
import json
import base64
import requests
import time
from datetime import datetime
from collections import defaultdict


# ================================================================
#  CONFIGURATION  -  edit these paths to match your machine
# ================================================================
FRAMES_DIR     = r"C:\Opeyemi\PROMPTS\FRAMES"   # pre-extracted frames
RESULTS_BASE   = r"C:\Opeyemi\PROMPTS\RESULTS"  # all JSON outputs
FRAME_EXT      = ".jpg"
FRAME_INTERVAL = 1   # 1=every frame; 2=every other; etc.
BATCH_SIZE     = 20   # max frames per API call

# ================================================================
#  FRAME HELPERS  -  self-contained in every cell
# ================================================================

def extract_frame_number(filename):
    """Return integer index from frame_00042.jpg style names."""
    import re as _re
    name = os.path.splitext(filename)[0]
    m = _re.search(r"frame[_\-]?(\d+)", name, _re.IGNORECASE)
    if m:
        return int(m.group(1))
    nums = _re.findall(r"\d+", name)
    return int(nums[-1]) if nums else 0


def discover_all_videos_and_frames(frames_dir=None):
    """
    Walk FRAMES_DIR and return a manifest of all extracted videos.

    Expected layout (created by the Video-to-Frames extractor):
        FRAMES_DIR/
            Abuse/
                Abuse001_x264/
                    frame_00001.jpg
                    frame_00002.jpg ...
                Abuse002_x264/ ...
            Arrest/ ...   (13 UCF-Crime categories)

    Returns dict "<CrimeType>_<VideoStem>" -> {
        "crime_type": "Abuse",
        "video_id":   "Abuse001_x264",
        "frames_dir": r"C:\...\FRAMES\Abuse\Abuse001_x264",
        "frames":     ["frame_00001.jpg", ...]   # sorted by number
    }
    """
    if frames_dir is None:
        frames_dir = FRAMES_DIR
    print(f"\n=== DISCOVERING FRAMES ===")
    print(f"    Root : {frames_dir}")
    all_videos = {}
    if not os.path.isdir(frames_dir):
        print(f"  ERROR: FRAMES_DIR not found: {frames_dir}")
        print("  Run the Video-to-Frames extractor first, or check the path.")
        return all_videos
    crime_types = sorted([
        d for d in os.listdir(frames_dir)
        if os.path.isdir(os.path.join(frames_dir, d)) and not d.startswith("_")
    ])
    print(f"  Categories : {crime_types}")
    for crime_type in crime_types:
        cat_dir = os.path.join(frames_dir, crime_type)
        video_stems = sorted([
            d for d in os.listdir(cat_dir)
            if os.path.isdir(os.path.join(cat_dir, d))
        ])
        print(f"    {crime_type:20s}: {len(video_stems)} videos")
        for video_stem in video_stems:
            vdir = os.path.join(cat_dir, video_stem)
            frame_files = sorted(
                [ff for ff in os.listdir(vdir) if ff.lower().endswith(FRAME_EXT)],
                key=extract_frame_number
            )
            if not frame_files:
                print(f"      WARNING: no {FRAME_EXT} frames in {vdir} - skipping")
                continue
            key = f"{crime_type}_{video_stem}"
            all_videos[key] = {
                "crime_type" : crime_type,
                "video_id"   : video_stem,
                "frames_dir" : vdir,
                "frames"     : frame_files,
            }
    print(f"  Total videos ready: {len(all_videos)}")
    return all_videos


def load_frames_for_video(video_info, frame_interval=1):
    """Read every frame_interval-th .jpg, base64-encode, return dict."""
    frames_data = {}
    vdir        = video_info["frames_dir"]
    frame_files = video_info["frames"]
    video_id    = video_info["video_id"]
    selected    = frame_files[::frame_interval]
    label = "ALL" if frame_interval == 1 else f"every {frame_interval}th"
    print(f"  Loading {len(selected)} frames ({label}) for {video_id} ...")
    for ff in selected:
        fp = os.path.join(vdir, ff)
        try:
            with open(fp, "rb") as fh:
                frames_data[ff] = base64.b64encode(fh.read()).decode("utf-8")
        except Exception as e:
            print(f"    ERROR loading {ff}: {e}")
    print(f"  Loaded {len(frames_data)}/{len(selected)} frames OK")
    return frames_data

SAVE_DIR = r"C:\Opeyemi\PROMPTS\RESULTS\GPT\SELF-CONSISTENCY"
os.makedirs(SAVE_DIR, exist_ok=True)


CHECKPOINT_FILE = os.path.join(SAVE_DIR, "self_consistency_checkpoint.json")


def load_checkpoint():
    """Resume from last completed video after any crash or network failure."""
    if os.path.exists(CHECKPOINT_FILE):
        try:
            with open(CHECKPOINT_FILE, "r") as f:
                data = json.load(f)
            n = len(data.get("completed_videos", []))
            print(f"  Checkpoint: {n} videos already done - skipping them.")
            return data
        except Exception as e:
            print(f"  Could not read checkpoint ({e}) - starting fresh.")
    return {"completed_videos": [], "results": {}}


def save_checkpoint(data):
    """Atomic write so the file is never corrupted on a crash."""
    os.makedirs(SAVE_DIR, exist_ok=True)
    tmp = CHECKPOINT_FILE + ".tmp"
    with open(tmp, "w") as f:
        json.dump(data, f, indent=2)
    os.replace(tmp, CHECKPOINT_FILE)







def make_gpt_request_robust(headers, payload, max_retries=7, base_wait=5):
    """
    Fault-tolerant OpenAI API call using requests.
    Retries on: rate limits (429), network errors, server errors (5xx).
    Exponential back-off with jitter, capped at 5 minutes.
    Returns parsed JSON on success, or error-string dict on permanent failure.
    """
    import random
    url = "https://api.openai.com/v1/chat/completions"

    for attempt in range(1, max_retries + 1):
        try:
            response = requests.post(url, headers=headers, json=payload, timeout=120)

            if response.status_code == 200:
                return response.json()

            # Rate limit
            if response.status_code == 429:
                wait = base_wait * (2 ** (attempt - 1)) + random.uniform(0, 2)
                wait = min(wait, 300)
                print(f"[Retry {attempt}/{max_retries}] 429 Rate limit. Waiting {wait:.1f}s ...")
                time.sleep(wait)
                continue

            # Server errors – retry
            if response.status_code >= 500:
                wait = base_wait * (2 ** (attempt - 1)) + random.uniform(0, 2)
                wait = min(wait, 300)
                print(f"[Retry {attempt}/{max_retries}] HTTP {response.status_code}. Waiting {wait:.1f}s ...")
                time.sleep(wait)
                continue

            # Permanent client errors (4xx except 429) – do not retry
            try:
                err = response.json()
            except Exception:
                err = response.text
            print(f"[FATAL] HTTP {response.status_code}: {err}")
            return {"error": f"HTTP_{response.status_code}", "detail": str(err)}

        except (requests.exceptions.ConnectionError,
                requests.exceptions.Timeout,
                requests.exceptions.ChunkedEncodingError) as e:
            wait = base_wait * (2 ** (attempt - 1)) + random.uniform(0, 2)
            wait = min(wait, 300)
            print(f"[Retry {attempt}/{max_retries}] Network error: {type(e).__name__}: {e}")
            print(f"  Waiting {wait:.1f}s ...")
            time.sleep(wait)

        except Exception as e:
            if attempt < max_retries:
                print(f"[Retry {attempt}/{max_retries}] Unexpected: {type(e).__name__}: {e}")
                time.sleep(base_wait * attempt)
            else:
                return {"error": str(e)}

    return {"error": f"All {max_retries} attempts exhausted"}




def test_gpt_api(api_key):
    """Test GPT API connection"""
    print("Testing GPT-5.5 API connection...")

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }

    payload = {
        "model": "gpt-5.5",
        "messages": [
            {
                "role": "user",
                "content": "Hello, can you respond with 'API connection successful'?"
            }
        ],
        "max_completion_tokens": 50
    }

    try:
        response = requests.post("https://api.openai.com/v1/chat/completions",
                               headers=headers, json=payload)

        if response.status_code == 200:
            result = result_json
            if "choices" in result:
                print("✓ GPT-5.5 API connection successful!")
                print(f"Response: {result['choices'][0]['message']['content']}")
                return True
        else:
            print(f"✗ API Error {response.status_code}: {response.text}")
            return False

    except Exception as e:
        print(f"✗ Connection error: {str(e)}")
        return False

def check_authentication():
    """Placeholder function to check authentication"""
    return True

def run():
    # Load API key from file
    with open(r"C:\Opeyemi\PROMPTS\API-KEYS\chatgpt.txt", "r") as _f:
        api_key = _f.read().strip()

    """Main execution function"""
    print("Self-Consistency Prompting Crime Video Analysis with GPT-5.5 - ALL Frames")
    print("="*75)
    print("Self-Consistency = Multiple independent analyses with consistency verification")
    print("="*75)

    # Test directory access first
    print("Testing directory access...")
    for path in [FRAMES_DIR, SAVE_DIR]:
        print(f"Path: {path}")
        print(f"  Exists: {os.path.exists(path)}")
        if os.path.exists(path):
            try:
                contents = os.listdir(path)
                print(f"  Contains {len(contents)} items")
                if contents:
                    print(f"  First few items: {contents[:3]}")
            except Exception as e:
                print(f"  Error accessing contents: {str(e)}")

    # Get API key
    try:
        api_key_path = "C:\Opeyemi\PROMPTS\API-KEYS\chatgpt.txt"
        print(f"Trying to load API key from: {api_key_path}")
        print(f"File exists: {os.path.exists(api_key_path)}")

        f = open(api_key_path, "r")
        api_key = f.read().strip()
        f.close()

        if not api_key:
            print("✗ Failed to load GPT API key: File is empty")
            return

        print("✓ Successfully loaded GPT API key")
        print(f"API key starts with: {api_key[:5]}...")

    except Exception as e:
        print(f"✗ Failed to load GPT API key: {str(e)}")
        return

    # Test GPT API connection
    if not test_gpt_api(api_key):
        print("✗ GPT-5.5 API test failed. Please check your API key and connection.")
        return

    # Check authentication
    if not check_authentication():
        print("✗ Authentication not completed.")
        return

    # Verify directories exist
    print("\nVerifying directories:")
    print(f"Frames directory exists: {os.path.exists(FRAMES_DIR)}")
    print(f"Save directory exists: {os.path.exists(SAVE_DIR)}")

    if not os.path.exists(FRAMES_DIR):
        print(f"✗ Frames directory not found: {FRAMES_DIR}")
        return

    # Create save directory if it doesn't exist
    os.makedirs(SAVE_DIR, exist_ok=True)

    # Process all crime folders
    results = process_all_crime_folders(api_key)

    # Print summary
    total_frames_processed = 0
    total_videos_processed = len(results)
    total_independent_runs = 0

    for video_id, video_results in results.items():
        if video_results and 'Self_Consistency_Analysis' in video_results:
            analysis = video_results['Self_Consistency_Analysis']
            total_frames_processed += analysis.get('valid_frames', 0)
            if 'consistency_results' in analysis and 'methodology' in analysis['consistency_results']:
                total_independent_runs += analysis['consistency_results']['methodology'].get('num_runs', 0)

    print("\n" + "="*75)
    print(f"SELF-CONSISTENCY PROMPTING ANALYSIS COMPLETE!")
    print(f"Videos processed: {total_videos_processed}")
    print(f"Total frames analyzed: {total_frames_processed}")
    print(f"Total independent runs: {total_independent_runs}")
    print(f"Model used: GPT-5.5")
    print(f"Analysis pattern: Multiple Independent → Consistency Check → Consensus")
    print("="*75)

if __name__ == "__main__":
    run()

Self-Consistency Prompting Crime Video Analysis with GPT-5.5 - ALL Frames
Self-Consistency = Multiple independent analyses with consistency verification
Testing directory access...
Path: C:\Opeyemi\PROMPTS\FRAMES
  Exists: True
  Contains 13 items
  First few items: ['Abuse', 'Assault', 'Burglary']
Path: C:\Opeyemi\PROMPTS\RESULTS\GPT\SELF-CONSISTENCY
  Exists: True
  Contains 0 items
Trying to load API key from: C:\Opeyemi\PROMPTS\API-KEYS\chatgpt.txt
File exists: True
✓ Successfully loaded GPT API key
API key starts with: sk-pr...
Testing GPT-5.5 API connection...
✗ Connection error: name 'result_json' is not defined
✗ GPT-5.5 API test failed. Please check your API key and connection.


#Meta-Prompting
- Meta-Prompting that processes all frames from crime videos. This technique is unique because it uses the AI to generate its own specialized prompts for analysis.

Meta-Prompting Approach
The Meta-Prompting technique follows this innovative process:

- Prompt Generation: Instead of using predefined prompts, the system asks the AI to create specialized prompts for analyzing video frames
- Prompt Application: These AI-generated prompts are then used to analyze the actual frames
- Meta-Synthesis: The system also generates a specialized synthesis prompt to combine all chunk analyses

Implementation Highlights

Two-Stage Meta-Prompting:

- First Stage: For each chunk of frames, generate a specialized analysis prompt
- Second Stage: For final synthesis, generate a specialized synthesis prompt
- Both stages use the AI to create task-specific prompts rather than using predefined ones


Complete Frame Processing:

- Processes all frames in chunks of 10 frames each
- Each chunk undergoes the full meta-prompting process independently


Specialized Prompt Design Process: Guides the AI to create prompts that focus on:

Step-by-step observation:
- Objective description before interpretation
- Attention to easily missed details
- Organizing observations into a coherent narrative
- Avoids including example responses in the generated prompts


Fallback Safety:

- If meta-prompting fails, falls back to a simple seed prompt
Ensures analysis can continue even if prompt generation has issues

In [6]:
import os
import json
import base64
import requests
import time
from datetime import datetime
from collections import defaultdict


import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
# ================================================================
#  CONFIGURATION  -  edit these paths to match your machine
# ================================================================
FRAMES_DIR     = r"C:\Opeyemi\PROMPTS\FRAMES"   # pre-extracted frames
RESULTS_BASE   = r"C:\Opeyemi\PROMPTS\RESULTS"  # all JSON outputs
FRAME_EXT      = ".jpg"
FRAME_INTERVAL = 1   # 1=every frame; 2=every other; etc.
BATCH_SIZE     = 20   # max frames per API call
MAX_WORKERS    = 4    # parallel videos processed at once (OpenAI rate limits depend on tier; raise carefully)

# ================================================================
#  FRAME HELPERS  -  self-contained in every cell
# ================================================================

def extract_frame_number(filename):
    """Return integer index from frame_00042.jpg style names."""
    import re as _re
    name = os.path.splitext(filename)[0]
    m = _re.search(r"frame[_\-]?(\d+)", name, _re.IGNORECASE)
    if m:
        return int(m.group(1))
    nums = _re.findall(r"\d+", name)
    return int(nums[-1]) if nums else 0


def discover_all_videos_and_frames(frames_dir=None):
    """
    Walk FRAMES_DIR and return a manifest of all extracted videos.

    Expected layout (created by the Video-to-Frames extractor):
        FRAMES_DIR/
            Abuse/
                Abuse001_x264/
                    frame_00001.jpg
                    frame_00002.jpg ...
                Abuse002_x264/ ...
            Arrest/ ...   (13 UCF-Crime categories)

    Returns dict "<CrimeType>_<VideoStem>" -> {
        "crime_type": "Abuse",
        "video_id":   "Abuse001_x264",
        "frames_dir": r"C:\...\FRAMES\Abuse\Abuse001_x264",
        "frames":     ["frame_00001.jpg", ...]   # sorted by number
    }
    """
    if frames_dir is None:
        frames_dir = FRAMES_DIR
    print(f"\n=== DISCOVERING FRAMES ===")
    print(f"    Root : {frames_dir}")
    all_videos = {}
    if not os.path.isdir(frames_dir):
        print(f"  ERROR: FRAMES_DIR not found: {frames_dir}")
        print("  Run the Video-to-Frames extractor first, or check the path.")
        return all_videos
    crime_types = sorted([
        d for d in os.listdir(frames_dir)
        if os.path.isdir(os.path.join(frames_dir, d)) and not d.startswith("_")
    ])
    print(f"  Categories : {crime_types}")
    for crime_type in crime_types:
        cat_dir = os.path.join(frames_dir, crime_type)
        video_stems = sorted([
            d for d in os.listdir(cat_dir)
            if os.path.isdir(os.path.join(cat_dir, d))
        ])
        print(f"    {crime_type:20s}: {len(video_stems)} videos")
        for video_stem in video_stems:
            vdir = os.path.join(cat_dir, video_stem)
            frame_files = sorted(
                [ff for ff in os.listdir(vdir) if ff.lower().endswith(FRAME_EXT)],
                key=extract_frame_number
            )
            if not frame_files:
                print(f"      WARNING: no {FRAME_EXT} frames in {vdir} - skipping")
                continue
            key = f"{crime_type}_{video_stem}"
            all_videos[key] = {
                "crime_type" : crime_type,
                "video_id"   : video_stem,
                "frames_dir" : vdir,
                "frames"     : frame_files,
            }
    print(f"  Total videos ready: {len(all_videos)}")
    return all_videos


def load_frames_for_video(video_info, frame_interval=1):
    """Read every frame_interval-th .jpg, base64-encode, return dict."""
    frames_data = {}
    vdir        = video_info["frames_dir"]
    frame_files = video_info["frames"]
    video_id    = video_info["video_id"]
    selected    = frame_files[::frame_interval]
    label = "ALL" if frame_interval == 1 else f"every {frame_interval}th"
    print(f"  Loading {len(selected)} frames ({label}) for {video_id} ...")
    for ff in selected:
        fp = os.path.join(vdir, ff)
        try:
            with open(fp, "rb") as fh:
                frames_data[ff] = base64.b64encode(fh.read()).decode("utf-8")
        except Exception as e:
            print(f"    ERROR loading {ff}: {e}")
    print(f"  Loaded {len(frames_data)}/{len(selected)} frames OK")
    return frames_data

SAVE_DIR = r"C:\Opeyemi\PROMPTS\RESULTS\GPT\META-PROMPTING"
os.makedirs(SAVE_DIR, exist_ok=True)


CHECKPOINT_FILE = os.path.join(SAVE_DIR, "meta_prompting_checkpoint.json")


def load_checkpoint():
    """Resume from last completed video after any crash or network failure."""
    if os.path.exists(CHECKPOINT_FILE):
        try:
            with open(CHECKPOINT_FILE, "r") as f:
                data = json.load(f)
            n = len(data.get("completed_videos", []))
            print(f"  Checkpoint: {n} videos already done - skipping them.")
            return data
        except Exception as e:
            print(f"  Could not read checkpoint ({e}) - starting fresh.")
    return {"completed_videos": [], "results": {}}


def save_checkpoint(data):
    """Atomic write so the file is never corrupted on a crash."""
    os.makedirs(SAVE_DIR, exist_ok=True)
    tmp = CHECKPOINT_FILE + ".tmp"
    with open(tmp, "w") as f:
        json.dump(data, f, indent=2)
    os.replace(tmp, CHECKPOINT_FILE)







def make_gpt_request_robust(headers, payload, max_retries=7, base_wait=5):
    """
    Fault-tolerant OpenAI API call using requests.
    Retries on: rate limits (429), network errors, server errors (5xx).
    Exponential back-off with jitter, capped at 5 minutes.
    Returns parsed JSON on success, or error-string dict on permanent failure.
    """
    import random
    url = "https://api.openai.com/v1/chat/completions"

    for attempt in range(1, max_retries + 1):
        try:
            response = requests.post(url, headers=headers, json=payload, timeout=120)

            if response.status_code == 200:
                return response.json()

            # Rate limit
            if response.status_code == 429:
                wait = base_wait * (2 ** (attempt - 1)) + random.uniform(0, 2)
                wait = min(wait, 300)
                print(f"[Retry {attempt}/{max_retries}] 429 Rate limit. Waiting {wait:.1f}s ...")
                time.sleep(wait)
                continue

            # Server errors – retry
            if response.status_code >= 500:
                wait = base_wait * (2 ** (attempt - 1)) + random.uniform(0, 2)
                wait = min(wait, 300)
                print(f"[Retry {attempt}/{max_retries}] HTTP {response.status_code}. Waiting {wait:.1f}s ...")
                time.sleep(wait)
                continue

            # Permanent client errors (4xx except 429) – do not retry
            try:
                err = response.json()
            except Exception:
                err = response.text
            print(f"[FATAL] HTTP {response.status_code}: {err}")
            return {"error": f"HTTP_{response.status_code}", "detail": str(err)}

        except (requests.exceptions.ConnectionError,
                requests.exceptions.Timeout,
                requests.exceptions.ChunkedEncodingError) as e:
            wait = base_wait * (2 ** (attempt - 1)) + random.uniform(0, 2)
            wait = min(wait, 300)
            print(f"[Retry {attempt}/{max_retries}] Network error: {type(e).__name__}: {e}")
            print(f"  Waiting {wait:.1f}s ...")
            time.sleep(wait)

        except Exception as e:
            if attempt < max_retries:
                print(f"[Retry {attempt}/{max_retries}] Unexpected: {type(e).__name__}: {e}")
                time.sleep(base_wait * attempt)
            else:
                return {"error": str(e)}

    return {"error": f"All {max_retries} attempts exhausted"}





def process_all_crime_folders(api_key):
    """
    Analyse every video in FRAMES_DIR.
    Already-completed videos are skipped (checkpoint/resume) so re-running
    after any failure picks up exactly where it stopped.
    """
    analyzer   = MetaPromptingAnalyzer(api_key)
    all_videos = discover_all_videos_and_frames()
    if not all_videos:
        print("No videos found! Verify FRAMES_DIR path."); return {}
    cp          = load_checkpoint()
    all_results = cp.get("results", {})
    done_set    = set(cp.get("completed_videos", []))
    skipped     = []
    total       = len(all_videos)
    remaining   = {k: v for k, v in all_videos.items() if k not in done_set}
    print(f"\nVideos: total={total} | done={len(done_set)} | remaining={len(remaining)}")
    # ================================================================
    #  PARALLEL VIDEO PROCESSING
    # ================================================================
    checkpoint_lock = threading.Lock()
    print_lock      = threading.Lock()

    def _process_one_video(vkey, vinfo):
        """Process a single video in its own thread. Checkpoint writes are locked."""
        with print_lock:
            print(f"\n  [START] {vkey}")
        try:
            frames = load_frames_for_video(vinfo, FRAME_INTERVAL)
            if not frames:
                return vkey, None, "no frames"
            res = analyzer.analyze_frames(frames, vinfo["video_id"], vinfo["crime_type"])
            with checkpoint_lock:
                all_results[vkey] = res
                done_set.add(vkey)
                save_checkpoint({"completed_videos": list(done_set), "results": all_results})
            with print_lock:
                print(f"  [DONE]  {vkey}  ({len(done_set)}/{total})")
            return vkey, res, None
        except Exception as e:
            with print_lock:
                print(f"  [ERROR] {vkey}: {e}")
            with checkpoint_lock:
                save_checkpoint({"completed_videos": list(done_set), "results": all_results})
            return vkey, None, f"error: {e}"

    print(f"\n  Launching ThreadPoolExecutor with {MAX_WORKERS} parallel workers ...")
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [executor.submit(_process_one_video, k, v) for k, v in remaining.items()]
        for fut in as_completed(futures):
            vkey, _, err = fut.result()
            if err:
                skipped.append(f"{vkey} ({err})")

    ts = time.strftime("%Y%m%d_%H%M%S")
    summary = os.path.join(SAVE_DIR, f"meta_prompting_summary_{ts}.json")
    with open(summary, "w") as f:
        json.dump(all_results, f, indent=2)
    if skipped:
        with open(os.path.join(SAVE_DIR, f"skipped_{ts}.txt"), "w") as f:
            f.write("\n".join(skipped))
    print(f"\nDone. Summary -> {summary}")
    print(f"  Processed: {len(all_results)} | Skipped: {len(skipped)}")
    return all_results


def test_gpt_api(api_key):
    """Test GPT API connection"""
    print("Testing GPT-5.5 API connection...")

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }

    payload = {
        "model": "gpt-5.5",
        "messages": [
            {
                "role": "user",
                "content": "Hello, can you respond with 'API connection successful'?"
            }
        ],
        "max_completion_tokens": 50
    }

    try:
        response = requests.post("https://api.openai.com/v1/chat/completions",
                               headers=headers, json=payload)

        if response.status_code == 200:
            result = result_json
            if "choices" in result:
                print("✓ GPT-5.5 API connection successful!")
                print(f"Response: {result['choices'][0]['message']['content']}")
                return True
        else:
            print(f"✗ API Error {response.status_code}: {response.text}")
            return False

    except Exception as e:
        print(f"✗ Connection error: {str(e)}")
        return False

def run():
    # Load API key from file
    with open(r"C:\Opeyemi\PROMPTS\API-KEYS\chatgpt.txt", "r") as _f:
        api_key = _f.read().strip()

    """Main execution function"""
    print("Meta-Prompting Crime Video Analysis with GPT-5.5 - ALL Frames")
    print("="*65)
    print("Meta-Prompting = Chunk-based meta-prompting with effectiveness evaluation")
    print("="*65)

    # Test directory access first
    print("Testing directory access...")
    for path in [FRAMES_DIR, SAVE_DIR]:
        print(f"Path: {path}")
        print(f"  Exists: {os.path.exists(path)}")
        if os.path.exists(path):
            try:
                contents = os.listdir(path)
                print(f"  Contains {len(contents)} items")
                if contents:
                    print(f"  First few items: {contents[:3]}")
            except Exception as e:
                print(f"  Error accessing contents: {str(e)}")

    # Get API key
    try:
        api_key_path = "C:\Opeyemi\PROMPTS\API-KEYS\chatgpt.txt"
        print(f"Trying to load API key from: {api_key_path}")
        print(f"File exists: {os.path.exists(api_key_path)}")

        f = open(api_key_path, "r")
        api_key = f.read().strip()
        f.close()

        if not api_key:
            print("✗ Failed to load GPT API key: File is empty")
            return

        print("✓ Successfully loaded GPT API key")
        print(f"API key starts with: {api_key[:5]}...")

    except Exception as e:
        print(f"✗ Failed to load GPT API key: {str(e)}")
        return

    # Test GPT API connection
    if not test_gpt_api(api_key):
        print("✗ GPT-5.5 API test failed. Please check your API key and connection.")
        return

    # Verify directories exist
    print("\nVerifying directories:")
    print(f"Frames directory exists: {os.path.exists(FRAMES_DIR)}")
    print(f"Save directory exists: {os.path.exists(SAVE_DIR)}")

    if not os.path.exists(FRAMES_DIR):
        print(f"✗ Frames directory not found: {FRAMES_DIR}")
        return

    # Create save directory if it doesn't exist
    os.makedirs(SAVE_DIR, exist_ok=True)

    # Process all crime folders
    results = process_all_crime_folders(api_key)

    # Print summary
    total_frames_processed = 0
    total_videos_processed = len(results)
    total_phases_completed = 0

    for video_id, video_results in results.items():
        if video_results and 'Meta_Prompting_Analysis' in video_results:
            analysis = video_results['Meta_Prompting_Analysis']
            total_frames_processed += analysis.get('valid_frames', 0)
            if 'meta_prompting_results' in analysis and 'methodology' in analysis['meta_prompting_results']:
                total_phases_completed += len(analysis['meta_prompting_results']['methodology'].get('phases', []))

    print("\n" + "="*65)
    print(f"META-PROMPTING ANALYSIS COMPLETE!")
    print(f"Videos processed: {total_videos_processed}")
    print(f"Total frames analyzed: {total_frames_processed}")
    print(f"Total meta-phases completed: {total_phases_completed}")
    print(f"Model used: GPT-5.5")
    print(f"Analysis pattern: Generate → Apply → Synthesize → Evaluate")
    print("="*65)

if __name__ == "__main__":
    run()

Meta-Prompting Crime Video Analysis with GPT-5.5 - ALL Frames
Meta-Prompting = Chunk-based meta-prompting with effectiveness evaluation
Testing directory access...
Path: C:\Opeyemi\PROMPTS\FRAMES
  Exists: True
  Contains 13 items
  First few items: ['Abuse', 'Assault', 'Burglary']
Path: C:\Opeyemi\PROMPTS\RESULTS\GPT\META-PROMPTING
  Exists: True
  Contains 0 items
Trying to load API key from: C:\Opeyemi\PROMPTS\API-KEYS\chatgpt.txt
File exists: True
✓ Successfully loaded GPT API key
API key starts with: sk-pr...
Testing GPT-5.5 API connection...
✗ Connection error: name 'result_json' is not defined
✗ GPT-5.5 API test failed. Please check your API key and connection.


#Chain-Of-Thought Prompting
Chain of Thought (CoT) prompting approach that processes all frames from crime videos. This technique explicitly encourages the model to show its

step-by-step reasoning process.
- Chain of Thought Prompting Approach: The Chain of Thought technique follows this explicit reasoning process:

Step-by-Step Reasoning: The approach explicitly asks the model to "think step by step" through its analysis
- Transparent Reasoning: Each reasoning step is clearly articulated in the response
- Structured Progression: The analysis follows a logical progression from observation to conclusion
- Reasoning Synthesis: The final synthesis also uses step-by-step reasoning to connect all segments

Implementation Highlights

Structured Reasoning Steps:

The prompt breaks down the analysis into 6 clear steps:

- Objective observation without interpretation
- Identification of key actors
- Chronological sequence of events
- Important objects and their usage
- Context and setting analysis
- Integration of observations into a coherent description


Each step builds on the previous one in a logical progression


Complete Frame Processing:

- Processes all frames in chunks of 10 frames each
- Each chunk undergoes the full chain of thought process independently


Reasoning-Based Synthesis: The synthesis prompt also follows a chain of thought structure:

- Extraction of key information from each segment
- Timeline construction across all segments
- Tracking people across multiple segments
- Tracking objects across segments
- Contextual integration of segments
- Construction of a comprehensive description


This ensures the synthesis uses the same reasoning approach as individual chunks


Explicit Prompting for Reasoning:

- Both the analysis and synthesis prompts specifically ask to "think step by step"
- System messages reinforce the importance of step-by-step reasoning
The model is explicitly asked to show its thinking process at each step

In [ ]:
import os
import json
import base64
import requests
import time
from datetime import datetime
from collections import defaultdict


import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
# ================================================================
#  CONFIGURATION  -  edit these paths to match your machine
# ================================================================
FRAMES_DIR     = r"C:\Opeyemi\PROMPTS\FRAMES"   # pre-extracted frames
RESULTS_BASE   = r"C:\Opeyemi\PROMPTS\RESULTS"  # all JSON outputs
FRAME_EXT      = ".jpg"
FRAME_INTERVAL = 1   # 1=every frame; 2=every other; etc.
BATCH_SIZE     = 20   # max frames per API call
MAX_WORKERS    = 4    # parallel videos processed at once (OpenAI rate limits depend on tier; raise carefully)

# ================================================================
#  FRAME HELPERS  -  self-contained in every cell
# ================================================================

def extract_frame_number(filename):
    """Return integer index from frame_00042.jpg style names."""
    import re as _re
    name = os.path.splitext(filename)[0]
    m = _re.search(r"frame[_\-]?(\d+)", name, _re.IGNORECASE)
    if m:
        return int(m.group(1))
    nums = _re.findall(r"\d+", name)
    return int(nums[-1]) if nums else 0


def discover_all_videos_and_frames(frames_dir=None):
    """
    Walk FRAMES_DIR and return a manifest of all extracted videos.

    Expected layout (created by the Video-to-Frames extractor):
        FRAMES_DIR/
            Abuse/
                Abuse001_x264/
                    frame_00001.jpg
                    frame_00002.jpg ...
                Abuse002_x264/ ...
            Arrest/ ...   (13 UCF-Crime categories)

    Returns dict "<CrimeType>_<VideoStem>" -> {
        "crime_type": "Abuse",
        "video_id":   "Abuse001_x264",
        "frames_dir": r"C:\...\FRAMES\Abuse\Abuse001_x264",
        "frames":     ["frame_00001.jpg", ...]   # sorted by number
    }
    """
    if frames_dir is None:
        frames_dir = FRAMES_DIR
    print(f"\n=== DISCOVERING FRAMES ===")
    print(f"    Root : {frames_dir}")
    all_videos = {}
    if not os.path.isdir(frames_dir):
        print(f"  ERROR: FRAMES_DIR not found: {frames_dir}")
        print("  Run the Video-to-Frames extractor first, or check the path.")
        return all_videos
    crime_types = sorted([
        d for d in os.listdir(frames_dir)
        if os.path.isdir(os.path.join(frames_dir, d)) and not d.startswith("_")
    ])
    print(f"  Categories : {crime_types}")
    for crime_type in crime_types:
        cat_dir = os.path.join(frames_dir, crime_type)
        video_stems = sorted([
            d for d in os.listdir(cat_dir)
            if os.path.isdir(os.path.join(cat_dir, d))
        ])
        print(f"    {crime_type:20s}: {len(video_stems)} videos")
        for video_stem in video_stems:
            vdir = os.path.join(cat_dir, video_stem)
            frame_files = sorted(
                [ff for ff in os.listdir(vdir) if ff.lower().endswith(FRAME_EXT)],
                key=extract_frame_number
            )
            if not frame_files:
                print(f"      WARNING: no {FRAME_EXT} frames in {vdir} - skipping")
                continue
            key = f"{crime_type}_{video_stem}"
            all_videos[key] = {
                "crime_type" : crime_type,
                "video_id"   : video_stem,
                "frames_dir" : vdir,
                "frames"     : frame_files,
            }
    print(f"  Total videos ready: {len(all_videos)}")
    return all_videos


def load_frames_for_video(video_info, frame_interval=1):
    """Read every frame_interval-th .jpg, base64-encode, return dict."""
    frames_data = {}
    vdir        = video_info["frames_dir"]
    frame_files = video_info["frames"]
    video_id    = video_info["video_id"]
    selected    = frame_files[::frame_interval]
    label = "ALL" if frame_interval == 1 else f"every {frame_interval}th"
    print(f"  Loading {len(selected)} frames ({label}) for {video_id} ...")
    for ff in selected:
        fp = os.path.join(vdir, ff)
        try:
            with open(fp, "rb") as fh:
                frames_data[ff] = base64.b64encode(fh.read()).decode("utf-8")
        except Exception as e:
            print(f"    ERROR loading {ff}: {e}")
    print(f"  Loaded {len(frames_data)}/{len(selected)} frames OK")
    return frames_data

SAVE_DIR = r"C:\Opeyemi\PROMPTS\RESULTS\GPT\CHAIN-OF-THOUGHT"
os.makedirs(SAVE_DIR, exist_ok=True)


CHECKPOINT_FILE = os.path.join(SAVE_DIR, "cot_checkpoint.json")


def load_checkpoint():
    """Resume from last completed video after any crash or network failure."""
    if os.path.exists(CHECKPOINT_FILE):
        try:
            with open(CHECKPOINT_FILE, "r") as f:
                data = json.load(f)
            n = len(data.get("completed_videos", []))
            print(f"  Checkpoint: {n} videos already done - skipping them.")
            return data
        except Exception as e:
            print(f"  Could not read checkpoint ({e}) - starting fresh.")
    return {"completed_videos": [], "results": {}}


def save_checkpoint(data):
    """Atomic write so the file is never corrupted on a crash."""
    os.makedirs(SAVE_DIR, exist_ok=True)
    tmp = CHECKPOINT_FILE + ".tmp"
    with open(tmp, "w") as f:
        json.dump(data, f, indent=2)
    os.replace(tmp, CHECKPOINT_FILE)








def _build_image_content(frames_data):
    """Convert frames_data dict into GPT vision content blocks, BATCH_SIZE at a time."""
    frame_names = sorted(frames_data.keys(), key=lambda x: extract_frame_number(x))
    batches = [frame_names[i:i+BATCH_SIZE] for i in range(0, len(frame_names), BATCH_SIZE)]
    return batches, frame_names


def _make_payload(api_key, messages):
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }
    payload = {
        "model": "gpt-5.5",
        "messages": messages,
        "max_completion_tokens": 4096
    }
    return headers, payload


def _frames_to_content(frames_data, frame_names):
    """Build a GPT vision content list from a batch of frame names."""
    content = []
    for fname in frame_names:
        b64 = frames_data[fname]
        content.append({
            "type": "image_url",
            "image_url": {
                "url": f"data:image/jpeg;base64,{b64}",
                "detail": "low"
            }
        })
        content.append({"type": "text", "text": f"[Frame: {fname}]"})
    return content


class ChainOfThoughtAnalyzer:
    """
    Chain-of-Thought crime analysis using GPT-5.5 with BATCH_SIZE frame batching.
    Steps 1-3 process frames in batches (images); steps 4-6 and final answer are text-only.
    """
    def __init__(self, api_key):
        self.api_key = api_key

    def _text_call(self, messages):
        headers, payload = _make_payload(self.api_key, messages)
        result = make_gpt_request_robust(headers, payload)
        return result.get("choices",[{}])[0].get("message",{}).get("content", str(result))

    def analyze_frames(self, frames_data, video_id, crime_type):
        print(f"\n  [Chain-of-Thought] {video_id} | {len(frames_data)} frames ...")
        frame_names = sorted(frames_data.keys(), key=lambda x: extract_frame_number(x))
        batches = [frame_names[i:i+BATCH_SIZE] for i in range(0, len(frame_names), BATCH_SIZE)]
        cot_steps = {}

        # Steps 1-3: Observe, Identify, Detect (images, batched)
        print(f"    Steps 1-3 (images): {len(batches)} batches ...")
        batch_summaries = []
        for idx, batch in enumerate(batches, 1):
            content = _frames_to_content(frames_data, batch)
            content.append({"type": "text", "text": (
                f"Steps 1-3 for Batch {idx}/{len(batches)}:\n"
                "STEP 1 - OBSERVE: List visible elements (setting, lighting, objects, people count).\n"
                "STEP 2 - IDENTIFY: Describe each person (appearance, clothing, position).\n"
                "STEP 3 - DETECT ACTIONS: Describe all movements, interactions, behaviours frame by frame."
            )})
            messages = [{"role": "user", "content": content}]
            headers, payload = _make_payload(self.api_key, messages)
            result = make_gpt_request_robust(headers, payload)
            s = result.get("choices",[{}])[0].get("message",{}).get("content", str(result))
            batch_summaries.append(s)
            print(f"      Batch {idx}: {len(s)} chars")

        formatted = "\n\n".join(f"--- Batch {i+1} ---\n{s}" for i,s in enumerate(batch_summaries))
        evidence = self._text_call([{"role": "user", "content": (
            f"Synthesize Steps 1-3 across {len(batches)} batches of {len(frames_data)} frames:\n{formatted}\n\n"
            "COMBINED STEP 1 (Observe): [unified scene]\n"
            "COMBINED STEP 2 (Identify): [all people]\n"
            "COMBINED STEP 3 (Actions): [all behaviours and events]"
        )}])
        cot_steps["steps123_batch_summaries"] = batch_summaries
        cot_steps["steps123_evidence_base"] = evidence
        print(f"      Evidence base: {len(evidence)} chars")

        for step_num, (label, prompt) in enumerate([
            ("step4_analyse", f"STEP 4 - ANALYSE PATTERNS:\n\nEvidence from Steps 1-3:\n{evidence}\n\nWhat patterns emerge? Which behaviours are suspicious? What are relationships between people? What is the likely intent?"),
            ("step5_classify", "STEP 5 - CLASSIFY:\n\nBased on Step 4 analysis, determine crime type. Consider each: Abuse, Arrest, Arson, Assault, Burglary, Explosion, Fighting, RoadAccidents, Robbery, Shooting, Shoplifting, Stealing, Vandalism, Normal. What evidence supports or rules out each?"),
            ("step6_verify", "STEP 6 - VERIFY REASONING:\n\nReview your Step 5 classification. Did I consider all evidence? Any logical gaps? Could I be wrong? Is confidence appropriate? Does the classification stand?"),
            ("final_answer", "FINAL ANSWER:\nPRIMARY CLASSIFICATION: [crime type]\nCONFIDENCE LEVEL: [0-100%]\nSEVERITY: [Low/Medium/High/Critical]\nREASONING CHAIN SUMMARY: [how each step led to conclusion]\nKEY EVIDENCE:\n- [point 1]\n- [point 2]\nALTERNATIVE INTERPRETATIONS: [other explanations]\nRECOMMENDED LAW ENFORCEMENT RESPONSE: [actions]"),
        ], start=4):
            print(f"    Step {step_num} / {label} ...")
            response = self._text_call([{"role": "user", "content": prompt}])
            cot_steps[label] = response
            print(f"      Response: {len(response)} chars")

        return {
            "video_id": video_id, "crime_type": crime_type,
            "frames_analyzed": len(frames_data), "total_batches": len(batches),
            "batch_size": BATCH_SIZE, "prompting_technique": "CHAIN-OF-THOUGHT",
            "model": "gpt-5.5", "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
            "cot_steps": cot_steps
        }



def make_gpt_request_robust(headers, payload, max_retries=7, base_wait=5):
    """
    Fault-tolerant OpenAI API call using requests.
    Retries on: rate limits (429), network errors, server errors (5xx).
    Exponential back-off with jitter, capped at 5 minutes.
    Returns parsed JSON on success, or error-string dict on permanent failure.
    """
    import random
    url = "https://api.openai.com/v1/chat/completions"

    for attempt in range(1, max_retries + 1):
        try:
            response = requests.post(url, headers=headers, json=payload, timeout=120)

            if response.status_code == 200:
                return response.json()

            # Rate limit
            if response.status_code == 429:
                wait = base_wait * (2 ** (attempt - 1)) + random.uniform(0, 2)
                wait = min(wait, 300)
                print(f"[Retry {attempt}/{max_retries}] 429 Rate limit. Waiting {wait:.1f}s ...")
                time.sleep(wait)
                continue

            # Server errors – retry
            if response.status_code >= 500:
                wait = base_wait * (2 ** (attempt - 1)) + random.uniform(0, 2)
                wait = min(wait, 300)
                print(f"[Retry {attempt}/{max_retries}] HTTP {response.status_code}. Waiting {wait:.1f}s ...")
                time.sleep(wait)
                continue

            # Permanent client errors (4xx except 429) – do not retry
            try:
                err = response.json()
            except Exception:
                err = response.text
            print(f"[FATAL] HTTP {response.status_code}: {err}")
            return {"error": f"HTTP_{response.status_code}", "detail": str(err)}

        except (requests.exceptions.ConnectionError,
                requests.exceptions.Timeout,
                requests.exceptions.ChunkedEncodingError) as e:
            wait = base_wait * (2 ** (attempt - 1)) + random.uniform(0, 2)
            wait = min(wait, 300)
            print(f"[Retry {attempt}/{max_retries}] Network error: {type(e).__name__}: {e}")
            print(f"  Waiting {wait:.1f}s ...")
            time.sleep(wait)

        except Exception as e:
            if attempt < max_retries:
                print(f"[Retry {attempt}/{max_retries}] Unexpected: {type(e).__name__}: {e}")
                time.sleep(base_wait * attempt)
            else:
                return {"error": str(e)}

    return {"error": f"All {max_retries} attempts exhausted"}





def process_all_crime_folders(api_key):
    """
    Analyse every video in FRAMES_DIR.
    Already-completed videos are skipped (checkpoint/resume) so re-running
    after any failure picks up exactly where it stopped.
    """
    analyzer   = ChainOfThoughtAnalyzer(api_key)
    all_videos = discover_all_videos_and_frames()
    if not all_videos:
        print("No videos found! Verify FRAMES_DIR path."); return {}
    cp          = load_checkpoint()
    all_results = cp.get("results", {})
    done_set    = set(cp.get("completed_videos", []))
    skipped     = []
    total       = len(all_videos)
    remaining   = {k: v for k, v in all_videos.items() if k not in done_set}
    print(f"\nVideos: total={total} | done={len(done_set)} | remaining={len(remaining)}")
    # ================================================================
    #  PARALLEL VIDEO PROCESSING
    # ================================================================
    checkpoint_lock = threading.Lock()
    print_lock      = threading.Lock()

    def _process_one_video(vkey, vinfo):
        """Process a single video in its own thread. Checkpoint writes are locked."""
        with print_lock:
            print(f"\n  [START] {vkey}")
        try:
            frames = load_frames_for_video(vinfo, FRAME_INTERVAL)
            if not frames:
                return vkey, None, "no frames"
            res = analyzer.analyze_frames(frames, vinfo["video_id"], vinfo["crime_type"])
            with checkpoint_lock:
                all_results[vkey] = res
                done_set.add(vkey)
                save_checkpoint({"completed_videos": list(done_set), "results": all_results})
            with print_lock:
                print(f"  [DONE]  {vkey}  ({len(done_set)}/{total})")
            return vkey, res, None
        except Exception as e:
            with print_lock:
                print(f"  [ERROR] {vkey}: {e}")
            with checkpoint_lock:
                save_checkpoint({"completed_videos": list(done_set), "results": all_results})
            return vkey, None, f"error: {e}"

    print(f"\n  Launching ThreadPoolExecutor with {MAX_WORKERS} parallel workers ...")
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [executor.submit(_process_one_video, k, v) for k, v in remaining.items()]
        for fut in as_completed(futures):
            vkey, _, err = fut.result()
            if err:
                skipped.append(f"{vkey} ({err})")

    ts = time.strftime("%Y%m%d_%H%M%S")
    summary = os.path.join(SAVE_DIR, f"cot_summary_{ts}.json")
    with open(summary, "w") as f:
        json.dump(all_results, f, indent=2)
    if skipped:
        with open(os.path.join(SAVE_DIR, f"skipped_{ts}.txt"), "w") as f:
            f.write("\n".join(skipped))
    print(f"\nDone. Summary -> {summary}")
    print(f"  Processed: {len(all_results)} | Skipped: {len(skipped)}")
    return all_results


def run():
    # Load API key from file
    with open(r"C:\Opeyemi\PROMPTS\API-KEYS\chatgpt.txt", "r") as _f:
        api_key = _f.read().strip()

    """Main execution function"""
    print("Chain of Thought Prompting Crime Video Analysis - ALL Frames")
    print("="*50)

    # Test directory access first
    print("Testing directory access...")
    for path in [FRAMES_DIR, SAVE_DIR]:
        print(f"Path: {path}")
        print(f"  Exists: {os.path.exists(path)}")
        if os.path.exists(path):
            try:
                contents = os.listdir(path)
                print(f"  Contains {len(contents)} items")
                if contents:
                    print(f"  First few items: {contents[:3]}")
            except Exception as e:
                print(f"  Error accessing contents: {str(e)}")

    # Get API key
    try:
        api_key_path = "C:\Opeyemi\PROMPTS\API-KEYS\chatgpt.txt"
        print(f"Trying to load API key from: {api_key_path}")
        print(f"File exists: {os.path.exists(api_key_path)}")

        f = open(api_key_path, "r")
        api_key = f.read().strip()
        f.close()

        if not api_key:
            print("✗ Failed to load GPT API key: File is empty")
            return

        print("✓ Successfully loaded GPT API key")
        print(f"API key starts with: {api_key[:5]}...")

    except Exception as e:
        print(f"✗ Failed to load GPT API key: {str(e)}")
        return

    # Verify directories exist
    print("\nVerifying directories:")
    print(f"Frames directory exists: {os.path.exists(FRAMES_DIR)}")
    print(f"Save directory exists: {os.path.exists(SAVE_DIR)}")

    if not os.path.exists(FRAMES_DIR):
        print(f"✗ Frames directory not found: {FRAMES_DIR}")
        return

    # Create save directory if it doesn't exist
    os.makedirs(SAVE_DIR, exist_ok=True)

    # Process all crime folders
    results = process_all_crime_folders(api_key)

    print("\n" + "="*50)
    print(f"CHAIN OF THOUGHT PROMPTING COMPLETE!")
    print(f"Videos processed: {len(results)}")
    print("="*50)

if __name__ == "__main__":
    run()

Chain of Thought Prompting Crime Video Analysis - ALL Frames
Testing directory access...
Path: C:\Opeyemi\PROMPTS\FRAMES
  Exists: True
  Contains 13 items
  First few items: ['Abuse', 'Assault', 'Burglary']
Path: C:\Opeyemi\PROMPTS\RESULTS\GPT\CHAIN-OF-THOUGHT
  Exists: True
  Contains 0 items
Trying to load API key from: C:\Opeyemi\PROMPTS\API-KEYS\chatgpt.txt
File exists: True
✓ Successfully loaded GPT API key
API key starts with: sk-pr...

Verifying directories:
Frames directory exists: True
Save directory exists: True

=== DISCOVERING FRAMES ===
    Root : C:\Opeyemi\PROMPTS\FRAMES
  Categories : ['Abuse', 'Assault', 'Burglary', 'Explosion', 'Fighting', 'RoadAccidents', 'Robbery', 'Shooting', 'Shoplifting', 'Stealing', 'Vandalism']
    Abuse               : 50 videos
    Assault             : 12 videos
    Burglary            : 100 videos
    Explosion           : 50 videos
    Fighting            : 50 videos
    RoadAccidents       : 150 videos
    Robbery             : 150 video